# 손동작 CAPTCHA 데이터셋 확장 파이프라인 (Colab 실행용)

`dataset_expansion_spec_v1.md` §1~6 기반, Jester 표본으로 가림 원 비율(w2_new/w3_new)을 재추정합니다.

**수행 순서**: 1) 저장소 클론 → 2) Jester 표본 업로드/마운트 → 3) Stage 1~7 실행 → 4) 결과 다운로드.

**주의**: Jester/InterHand2.6M은 연구용 라이선스가 있는 데이터셋입니다. 공식 사이트에서 본인 계정으로 가입/동의하신 후 받은 파일만 사용하세요. 전체를 다 받을 필요 없으며, 수백~수천 클립 정도의 표본이면 충분합니다.

In [ ]:
#셀 1 - 의존성 설치
!pip install -q mediapipe==0.10.14 opencv-python-headless numpy

In [ ]:
#셀 2 - 저장소 클론 (dataset_pipeline 코드 받기)
# dataset_pipeline드가 GitHub에 업로드된 뒤에만 이 셀이 작동합니다.
!rm -rf handcaptcha
!git clone --depth 1 https://github.com/mukgore/handcaptcha.git
import sys
sys.path.insert(0, '/content/handcaptcha')

import os
assert os.path.isdir('/content/handcaptcha/dataset_pipeline'), \
    'dataset_pipeline이 저장소에 없습니다 - 먼저 GitHub에 업로드해주세요.'
print('dataset_pipeline 로드 완료')

In [ ]:
#셀 3-A - Jester 표본 업로드 (소량 테스트용, 수십~수백 개 정도)
# 업로드 창이 뜨면 mp4 파일들을 여러 개 선택해서 올려주세요.
from google.colab import files
import os, shutil

os.makedirs('/content/jester_sample', exist_ok=True)
print('Jester 표본 mp4 파일들을 선택해주세요 (여러 개 선택 가능):')
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join('/content/jester_sample', name))
print(f"{len(uploaded)}개 파일 업로드 완료 → /content/jester_sample")

In [ ]:
#셀 3-B - (대안) Google Drive에 미리 올려둔 대용량 표본 마운트
# 수백~수천 개처럼 많을 때는 이 셀을 쓰세요 (셀 3-A는 건너뛰기).
# Drive의 '/MyDrive/jester_sample/' 폴더에 mp4들을 미리 넣어두세요.
# from google.colab import drive
# drive.mount('/content/drive')
# JESTER_SAMPLE_DIR = '/content/drive/MyDrive/jester_sample'

In [ ]:
#셀 4 - Stage 1~7 파이프라인 실행
import glob
from dataset_pipeline.pipeline import run_batch
from dataset_pipeline.config import PipelineConfig

JESTER_SAMPLE_DIR = '/content/jester_sample'  # 셀 3-B를 쓨다면 이 경로를 Drive 경로로 바꿔주세요.

video_paths = sorted(glob.glob(f'{JESTER_SAMPLE_DIR}/*.mp4'))
print(f'{len(video_paths)}개 클립 발견')

config = PipelineConfig()  # 필요하면 여기서 t_frontal_deg, t_straight_deg 등 오버라이드 가능
result = run_batch(video_paths, config=config, output_dir='/content/pipeline_output')

print(f"처리: {len(result['clip_reports'])}개 클립, 채택: {len(result['manifest'])}개")
print('재추정 가중치:', result['weights_report'])

In [ ]:
#셀 5 - 결과 다운로드 (pipeline_report.json / weights_report.json / manifest.json)
import shutil
from google.colab import files

shutil.make_archive('/content/pipeline_output', 'zip', '/content/pipeline_output')
files.download('/content/pipeline_output.zip')
print('다운로드 시작! 이 zip을 Claude에게 다시 전달하면 코드에 새 가중치를 반영해드립니다.')

## (선택) 셀 6 - §4 T_frontal 그리드서치
정면성 판정 임계값(T_frontal)을 실제 데이터로 찾고 싶을 때만 따로 실행하세요 (수백~수천 프레임 정도 필요, 시간이 더 걸립니다).

In [ ]:
#셀 6 - T_frontal 그리드서치 (선택사항)
from dataset_pipeline.grid_search_frontal import run_grid_search

grid_result = run_grid_search(video_paths, output='/content/t_frontal_grid_search.json')
print(grid_result)

from google.colab import files
files.download('/content/t_frontal_grid_search.json')